DATASET YANG SUDAH DIOLAH

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# =========================================================================
# 1. GENERASI DATASET (Sesuai Perintah: 100 Pos, 100 Neg, 100 Neu)
# =========================================================================

def create_structured_dataset():
    # Komponen kalimat untuk variasi data sintetis
    subjects = ["Produk", "Layanan", "Aplikasi", "Barang", "Makanan", "Kurir", "Admin", "Fitur", "Toko", "Kemasan"]

    pos_adj = ["sangat bagus", "luar biasa", "memuaskan", "recomended banget", "berkualitas tinggi", "mantap", "cepat sekali", "ramah", "keren", "oke"]
    neg_adj = ["buruk sekali", "mengecewakan", "rusak", "lambat", "kasar", "mahal tapi jelek", "error terus", "tidak sesuai", "payah", "parah"]
    neu_adj = ["biasa saja", "standard", "warna putih", "ukuran sedang", "sudah diterima", "sampai tadi siang", "cek dulu", "sedang dicoba", "ada di meja", "informasi lanjut"]

    # Menghasilkan 100 data per kategori
    data_latih = []
    for _ in range(100):
        data_latih.append({"text": f"{np.random.choice(subjects)} ini {np.random.choice(pos_adj)}.", "label": 2}) # 2: Positif
    for _ in range(100):
        data_latih.append({"text": f"{np.random.choice(subjects)} itu {np.random.choice(neg_adj)}.", "label": 0}) # 0: Negatif
    for _ in range(100):
        data_latih.append({"text": f"{np.random.choice(subjects)} tersebut {np.random.choice(neu_adj)}.", "label": 1}) # 1: Netral

    train_df = pd.DataFrame(data_latih)

    # Menghasilkan 400 data pengujian (Tanpa Label)
    all_adj = pos_adj + neg_adj + neu_adj
    data_uji = []
    for _ in range(400):
        data_uji.append({"text": f"{np.random.choice(subjects)} {np.random.choice(all_adj)}."})

    test_df = pd.DataFrame(data_uji)

    return train_df, test_df

print("--- Menyiapkan Dataset ---")
train_df, test_df = create_structured_dataset()

# =========================================================================
# 2. PERSIAPAN TRANSFORMER (IndoBERT)
# =========================================================================

model_name = "indobenchmark/indobert-base-p2"
tokenizer = BertTokenizer.from_pretrained(model_name)

# Konfigurasi Dataset untuk PyTorch
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Split data latih menjadi Train & Validation (80:20) untuk evaluasi performa
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# Tokenisasi
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=64)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=64)

train_dataset = SentimentDataset(train_encodings, y_train)
val_dataset = SentimentDataset(val_encodings, y_val)

# Load Model
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)

# =========================================================================
# 3. TRAINING (FINE-TUNING)
# =========================================================================

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,              # Meningkatkan epoch untuk dataset kecil
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",            # Perbaikan: Menggunakan 'eval_strategy' bukan 'evaluation_strategy'
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\n--- Memulai Pelatihan Model ---")
trainer.train()

# =========================================================================
# 4. PELABELAN DATA PENGUJIAN (PREDIKSI)
# =========================================================================

print("\n--- Melakukan Pelabelan Otomatis pada 400 Data Pengujian ---")

test_encodings = tokenizer(test_df['text'].tolist(), truncation=True, padding=True, max_length=64)
test_dataset = SentimentDataset(test_encodings)

# Prediksi
raw_preds = trainer.predict(test_dataset)
predictions = np.argmax(raw_preds.predictions, axis=1)

# Map Label
label_map = {0: "Negatif", 1: "Netral", 2: "Positif"}
test_df['label_hasil_prediksi'] = [label_map[p] for p in predictions]

# Tampilkan 10 baris pertama
print("\nSamples Hasil Pelabelan:")
print(test_df.head(10))

# Simpan ke CSV
output_file = 'hasil_sentimen.csv'
test_df.to_csv(output_file, index=False)
print(f"\nSelesai! Hasil pelabelan disimpan di: {output_file}")

--- Menyiapkan Dataset ---


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Memulai Pelatihan Model ---


Epoch,Training Loss,Validation Loss,Accuracy
1,1.079900,0.585919,0.916667
2,0.136000,0.014472,1.000000
3,0.008300,0.001570,1.000000
4,0.001100,0.000771,1.000000
5,0.000800,0.000647,1.000000



--- Melakukan Pelabelan Otomatis pada 400 Data Pengujian ---



Samples Hasil Pelabelan:
                        text label_hasil_prediksi
0          Makanan standard.              Positif
1           Kurir memuaskan.              Positif
2      Admin sudah diterima.              Positif
3  Kemasan mahal tapi jelek.              Negatif
4              Fitur mantap.              Positif
5       Fitur sedang dicoba.              Positif
6             Makanan ramah.              Positif
7     Kemasan sedang dicoba.              Positif
8  Barang sampai tadi siang.               Netral
9   Fitur sampai tadi siang.              Positif

Selesai! Hasil pelabelan disimpan di: hasil_sentimen.csv


DATASET YANG BELUM DIOLAH

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from sklearn.metrics import accuracy_score

# ==========================================
# 1. PERSIAPAN DATASET (300 LATIH, 400 UJI)
# ==========================================

def prepare_data():
    # Template kalimat untuk variasi data
    pos_samples = [
        "Produk ini sangat bagus.", "Sangat puas dengan layanannya.", "Luar biasa, sangat cepat.",
        "Terima kasih atas bantuannya.", "Kualitas barang jempolan.", "Sangat merekomendasikan toko ini.",
        "Aplikasi mudah digunakan.", "Fitur ini sangat membantu.", "Senang belanja di sini.", "Packing rapi."
    ]
    neg_samples = [
        "Produk rusak saat sampai.", "Sangat kecewa dengan respon admin.", "Pengiriman sangat lambat.",
        "Barang tidak sesuai deskripsi.", "Kualitas buruk sekali.", "Jangan beli di sini.",
        "Aplikasi sering error.", "Sangat menyesal membeli ini.", "Pelayanan kasar.", "Harga terlalu mahal."
    ]
    neu_samples = [
        "Barang sudah diterima.", "Warna produk adalah biru.", "Saya membeli ini kemarin.",
        "Status pesanan sedang dikirim.", "Berat paket satu kilo.", "Ada varian warna lain?",
        "Sedang mencoba aplikasinya.", "Lokasi toko di Jakarta.", "Pembayaran via transfer.", "Isi paket lengkap."
    ]

    # Menghasilkan 100 data per kategori untuk pelatihan (Total 300)
    train_data = []
    for _ in range(100):
        train_data.append({"text": np.random.choice(pos_samples), "label": 2}) # 2: Positif
        train_data.append({"text": np.random.choice(neg_samples), "label": 0}) # 0: Negatif
        train_data.append({"text": np.random.choice(neu_samples), "label": 1}) # 1: Netral

    train_df = pd.DataFrame(train_data)

    # Menghasilkan 400 data pengujian tanpa label
    all_texts = pos_samples + neg_samples + neu_samples
    test_data = [{"text": np.random.choice(all_texts)} for _ in range(400)]
    test_df = pd.DataFrame(test_data)

    return train_df, test_df

print("Mempersiapkan dataset...")
train_df, test_df = prepare_data()

# ==========================================
# 2. KONFIGURASI MODEL INDOBERT
# ==========================================

model_name = "indobenchmark/indobert-base-p2"
tokenizer = BertTokenizer.from_pretrained(model_name)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

# Tokenisasi
train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, padding=True, max_length=64)
train_dataset = SentimentDataset(train_encodings, train_df['label'].tolist())

# Load Model untuk 3 kelas
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)

# ==========================================
# 3. PROSES PELATIHAN (TRAINING)
# ==========================================

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print("Melatih model Transformer (Fine-tuning IndoBERT)...")
trainer.train()

# ==========================================
# 4. PELABELAN DATA PENGUJIAN
# ==========================================

print("Melakukan pelabelan pada 400 data uji...")
test_encodings = tokenizer(test_df['text'].tolist(), truncation=True, padding=True, max_length=64)
test_dataset = SentimentDataset(test_encodings)

# Prediksi makna
raw_preds = trainer.predict(test_dataset)
predicted_labels = np.argmax(raw_preds.predictions, axis=1)

# Mapping hasil ke teks
label_map = {0: "Negatif", 1: "Netral", 2: "Positif"}
test_df['label_hasil_prediksi'] = [label_map[l] for l in predicted_labels]

# Simpan hasil
output_filename = "hasil_pelabelan_transformer.csv"
test_df.to_csv(output_filename, index=False)

print(f"\nProses Selesai! Hasil pelabelan disimpan ke {output_filename}")
print("\nCuplikan Hasil:")
print(test_df.head(10))

Mempersiapkan dataset...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Melatih model Transformer (Fine-tuning IndoBERT)...


Step,Training Loss
10,0.988400
20,0.119900
30,0.005300
40,0.001700
50,0.001000
60,0.000800
70,0.000800
80,0.000700
90,0.000600
100,0.000600


Melakukan pelabelan pada 400 data uji...



Proses Selesai! Hasil pelabelan disimpan ke hasil_pelabelan_transformer.csv

Cuplikan Hasil:
                             text label_hasil_prediksi
0       Kualitas barang jempolan.              Positif
1  Status pesanan sedang dikirim.               Netral
2        Pembayaran via transfer.               Netral
3          Aplikasi sering error.              Negatif
4  Barang tidak sesuai deskripsi.              Negatif
5          Barang sudah diterima.               Netral
6       Luar biasa, sangat cepat.              Positif
7         Lokasi toko di Jakarta.               Netral
8        Produk ini sangat bagus.              Positif
9            Harga terlalu mahal.              Negatif
